#### 감염농장
##### 지오코딩: farm_address → lat/lng (카카오 주소검색 API)

In [ ]:
import sys
sys.path.append("/Workspace/방역로/00_Shared_Utils")
from utils_config import CATALOG, KAKAO_API_KEY
CATALOG = "dt4_team1_databricks"

import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, StringType

In [ ]:
# UDF 안에서 직접 키 읽기 (클로저 캡처 문제 회피)
def geocode(address: str):
    """주소 → (lat, lng), 실패 시 (None, None) / UDF 워커에서 직접 import"""

    res = requests.get(
        "https://dapi.kakao.com/v2/local/search/address.json",
        headers={"Authorization": f"KakaoAK {KAKAO_API_KEY}"},
        params={"query": address}
    )
    docs = res.json().get("documents", [])
    if not docs:
        return (None, None)
    return (float(docs[0]["y"]), float(docs[0]["x"]))  # lat, lng

geocode_udf = F.udf(
    lambda addr: geocode(addr),
    StructType([StructField("latitude", DoubleType()), StructField("longitude", DoubleType())])
)

In [ ]:
df_raw = spark.table(f"{CATALOG}.raw.outbreak")

df_silver = (
    df_raw
    .filter(F.col("OCCRRNC_DE").isNotNull())          # 발생일 없는 행 제거
    .select(
        F.split(F.col("FARM_LOCPLC"), " ")[1].alias("county"),        # 시군구 추출
        F.col("FARM_NM").alias("farm_name"),
        # "-" 기준 분리: "닭-산란계" → name=닭, type=산란계 / "오리" → name=오리, type=오리
        F.split(F.col("LVSTCKSPC_NM"), "-")[0].alias("livestock_name"),
        F.when(
            F.col("LVSTCKSPC_NM").contains("-"),
            F.split(F.col("LVSTCKSPC_NM"), "-")[1]
        ).otherwise(F.col("LVSTCKSPC_NM")).alias("livestock_type"),
        F.col("OCCRRNC_LVSTCKCNT").cast("int").alias("head_count"),
        F.col("FARM_LOCPLC").alias("farm_address"),
        F.to_date(F.col("OCCRRNC_DE"), "yyyyMMdd").alias("outbreak_date"),
        F.col("LVSTCKSPC_CODE").alias("livestock_code"),
        F.col("FARM_LOCPLC"),
    )
    .withColumn("_coords", geocode_udf(F.col("farm_address")))        # 카카오 API 호출
    .withColumn("latitude",  F.col("_coords.latitude"))
    .withColumn("longitude", F.col("_coords.longitude"))
    .drop("_coords")
)


In [ ]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.silver.infection_farm")

print(spark.table(f"{CATALOG}.silver.infection_farm").count())